## Feature Selection
* sklearn.feature_selection module has useful APIs to select features/ reduce dimensionality, either to improve estimators' accuracy scores or to boost their performance on very high-dimensional datasets.

### 1 Filter Based Methods

### 1.1 📊 VarianceThreshold

This transformer helps to keep only high variance features by providing a certain threshold. 
Features with a variance greater than or equal to the threshold value are kept; the rest are removed.

By default, it removes any feature with the same value (i.e., $0$ variance).

---


In [29]:
from sklearn.feature_extraction import DictVectorizer
import numpy as np
import pandas as pd

data = [
    {'age': 4, 'height':96.0},
    {'age': 1, 'height' :73.9},
    {'age': 3, 'height':88.9},
    {'age': 2, 'height' : 81.6} 
]

dv = DictVectorizer(sparse=False)

data_transformed = dv.fit_transform(data)

np.var(data_transformed , axis=0)

array([ 1.25 , 67.735])

In [30]:
from sklearn.feature_selection import VarianceThreshold

# Har column ka variance check karta hai jiska var threshold se jada rehta hai us column ke sare element ko select karta hai banki column ko drop 

vt = VarianceThreshold(threshold=9)
data_new = vt.fit_transform(data_transformed)
data_new

array([[96. ],
       [73.9],
       [88.9],
       [81.6]])

### 1.2 SelectKBest

### 🏆 Feature Selection: Understanding `SelectKBest`

### 🤔 1. Ye kya hai aur kyu ho raha hai? (What & Why)

Jab humare paas bohot saare columns hote hain, toh zaroori nahi ki sabhi columns result predict karne me madad karein. 
**Example:** Agar tum kisi Ghar ka price (`y`) predict kar rahe ho, toh 'Number of Bedrooms' bohot important feature hai, par 'Ghar ka color' shayad utna important na ho.

**`SelectKBest`** ek aisa smart filter hai jo sirf **Top 'K'** (sabse best) features ko chunta hai aur baaki sab ko drop kar deta hai. 

* **Kaise chunta hai?:** Yeh ek "Scoring Function" (jaise `mutual_info_regression` ya `f_regression`) ka use karta hai. Yeh function har column ka ek test leta hai ki wo target (`y`) ke sath kitna strongly connected hai, aur usko ek 'Score' deta hai.
* **Result:** Jinke score sabse high hote hain, wo Top 'K' features select ho jate hain.

---

### 🛠️ 2. Libraries, Packages & Modules Used

Aao is code ke tools ko samajhte hain:

1. **`sklearn.datasets` -> `fetch_california_housing`**
   * Yeh Scikit-Learn ka ek built-in dataset hai. Isme California ke alag-alag areas ki housing properties (jaise income, age, rooms) aur unke Prices diye gaye hain.

2. **`sklearn.feature_selection` -> `SelectKBest` (Class)**
   * Yeh humara **VIP Bouncer** hai. Isko tum bataoge ki "Bhai, mujhe sirf Top 3 (k=3) VIPs (features) chahiye", aur yeh baaki sabko bahar nikal dega.

3. **`sklearn.feature_selection` -> `mutual_info_regression` (Function)**
   * Yeh bouncer ki **"Checking Machine"** hai. Yeh function target (`y`) aur feature (`X`) ke beech ka "Mutual Information" nikalta hai. 
   * **Simple terms me:** Yeh check karta hai ki agar main feature ko badalun, toh usse target par kitna asar padega. Jitna zyada asar, utna bada score! (Isliye ise *regression* problems me use karte hain kyu ki target continuous/number hota hai).

---

In [31]:
from sklearn.datasets import fetch_california_housing
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# 1. Data Download kiya
X_california, y_california = fetch_california_housing(return_X_y=True)

# 2. Sirf pehli 2000 rows select ki (for faster processing)
X, y = X_california[:2000, :], y_california[:2000]

print(f"Shape of feature matrix before: {X.shape}")
# OUTPUT: (2000, 8) 
# Matlab humare paas 2000 rows aur 8 alag-alag features (columns) hain.


skb = SelectKBest(mutual_info_regression , k=3)
x_new = skb.fit_transform(X,y)
print(f'Shape of of feature matrix after feature selection: {x_new. shape}')

skb.get_feature_names_out()

Shape of feature matrix before: (2000, 8)
Shape of of feature matrix after feature selection: (2000, 3)


array(['x0', 'x6', 'x7'], dtype=object)

### 1.3 📈 Feature Selection: `SelectPercentile` & `get_feature_names_out`

## 🤔 1. Ye kya hai aur iski zaroorat kyu padi? (What & Why)

Humne pichle topic me `SelectKBest` padha tha jisme hum bolte the "Bhai sirf Top 3 features de do". Par socho agar kal ko tumhare dataset me 1000 naye columns aagaye? Tab bhi wo sirf 3 hi nikalega, jo galat ho sakta hai!

Is problem ko solve karta hai **`SelectPercentile`**.
* **Kya karta hai:** Yahan hum exact number (K) nahi batate, balki **Percentage** batate hain. Hum bolte hain, "Total jitne bhi features hain, unme se Top 30% best features nikal lo".
* **Fayda:** Yeh **Dynamic** hai. Agar 10 columns hain toh top 3 nikalega. Agar kal 100 columns ho gaye toh automatically top 30 nikal lega!

---

## 🛠️ 2. Naye Methods aur Tools (Packages)

Is code me humne 2 nayi cheezein dekhi hain:

1. **`sklearn.feature_selection` -> `SelectPercentile` (Class)**
   * Yeh humara "Percentage-based VIP Filter" hai. Scoring function wahi purana `mutual_info_regression` use ho raha hai.

2. **`.get_feature_names_out()` (Method / Function)**
   * **Bohot Important:** Jab Numpy kisi data ko filter karta hai, toh wo column ke naam (headers) bhool jata hai aur usko ek plain matrix bana deta hai. 
   * Yeh method hume batata hai ki jo features (columns) bach gaye, unke **Asli naam ya index kya the**.

---

In [32]:
from sklearn.feature_selection import SelectPercentile , mutual_info_regression

sp = SelectPercentile(mutual_info_regression)

x_new = sp.fit_transform(X,y)

print(f'Shape of of feature matrix after feature selection: {x_new. shape}')

Shape of of feature matrix after feature selection: (2000, 1)


### 1.3 🧰 Feature Selection: Understanding `GenericUnivariateSelect`

## 🤔 1. Ye kya hai aur iski zaroorat kyu padi? (What & Why)

Abhi tak humne Top 3 features nikalne ke liye `SelectKBest` use kiya, aur Top 30% nikalne ke liye `SelectPercentile` use kiya. 
Scikit-Learn walo ne socha, "Yaar, har choti cheez ke liye alag-alag naam (classes) yaad rakhna kitna mushkil hai!"

Toh unhone ek **Master Tool** banaya: **`GenericUnivariateSelect`**.
* **Kya karta hai:** Yeh akela tool `SelectKBest`, `SelectPercentile`, aur baaki sabhi univariate filters ka kaam kar sakta hai. 
* **Kaise karta hai:** Tumhe bas isko ek **`mode`** (tarika) batana padta hai aur uska **`param`** (value) dena padta hai. Yeh turant apna roop badal leta hai!

---

## 🛠️ 2. The "Modes" (Iske alag-alag roop)

Is tool me `mode` parameter ke zariye tum 5 alag-alag strategies apply kar sakte ho:

1. **`mode = 'k_best'`**: Yeh bilkul `SelectKBest` ban jayega. (Sath me `param=3` de do).
2. **`mode = 'percentile'`**: Yeh bilkul `SelectPercentile` ban jayega. (Sath me `param=30` de do).
3. **`mode = 'fpr'`**: False Positive Rate. Yeh un features ko nikalta hai jo statistical test me fail ho jate hain (p-value based).
4. **`mode = 'fdr'`**: False Discovery Rate. Yeh thoda advanced statistical filter hai.
5. **`mode = 'fwe'`**: Family Wise Error rate. Yeh aur bhi zyada strict statistical filter hai.

*(Note: Machine Learning me sabse zyada `k_best` aur `percentile` hi use hote hain!)*

---

In [ ]:
from sklearn.feature_selection import GenericUnivariateSelect

gus1 = GenericUnivariateSelect(mode="k_best" , param=3)
gus2 = GenericUnivariateSelect(mode="percentile", param=30)
gus3 = GenericUnivariateSelect(mode="fdr")
gus4 = GenericUnivariateSelect(mode="fwe")

x_new1 = gus1.fit_transform(X,y)
x_new2 = gus2.fit_transform(X,y)
x_new3 = gus3.fit_transform(X,y)
x_new4 = gus4.fit_transform(X,y)



## 2 Wrapper Based Method

### 2.1 REF (Recursive Reature Elimination)
* step 1 : Fits a model
* step 2 : Ranks the features , afterwards it removes one or more features(dependa upon `step` parameters)

In [46]:
from sklearn.datasets import make_friedman1
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

X, y = make_friedman1(n_samples=50, n_features=10, random_state=42)

estimator = LinearRegression()
selector = RFE(estimator , n_features_to_select=3 , step=1)
selector = selector.fit(X,y)

print(selector.support_)

print(f"Rank of Each Feature is : {selector.ranking_}")

[ True  True False  True False False False False False False]
Rank of Each Feature is : [1 1 5 1 2 3 7 4 8 6]


### 2.2 🎯 Feature Selection: `SelectFromModel`

## 🤔 1. Ye kya hai aur RFE se alag kaise hai? (What & Why)

**`SelectFromModel`** bhi ek machine learning model (jaise Linear Regression ya Random Forest) ka use karke features select karta hai. Par iska tarika `RFE` se alag hai.

*   **RFE ka tarika:** Model train karo -> sabse weak feature nikalo -> phir se naya model train karo. (Bohot slow).
*   **SelectFromModel ka tarika ("One-Shot"):** Model ko sirf **EK BAAR** train karo. Model har feature ko ek "Weight" (coefficient) de dega. Ab jiska weight zyada hai, usko rakh lo, baaki sabko ek sath nikal do! (Bohot fast).

---

## 🛠️ 2. Important Terms in Code

Is code me 2 nayi aur bohot important baatein hain:

1. **`estimator.coef_` (Coefficients):** 
   Jab Linear Regression model train hota hai, toh wo har feature ko ek multiplier (weight) deta hai. 
   * Agar weight bada positive number hai (e.g., +3.64): Matlab yeh feature target badhane me bohot important hai.
   * Agar weight bada negative number hai (e.g., -0.16): Matlab yeh feature target ghatane me bohot important hai. (Isiliye magnitude ya absolute value dekhna zaroori hai!)

2. **`prefit=True`:** 
   Agar tumne model ko pehle hi `fit()` (train) kar liya hai, toh `SelectFromModel` ko bolna padta hai: *"Bhai, maine model train kar diya hai, tu wapas train mat karna, seedha use kar."* Iske liye `prefit=True` lagate hain.

---



In [ ]:
import numpy as np
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

# 1. Judge (Model) bulaya aur usko Pura Data padha diya (Training)
estimator = LinearRegression()
estimator.fit(X, y)

# 2. Check karte hain judge ne kisko kya marks (coefficients) diye
print(f'Coefficients of features:\n{estimator.coef_}')

# 3. 'SelectFromModel' ko set up kiya
# max_features=3 -> Sirf top 3 chahiye
# prefit=True -> Model pehle se trained (fitted) hai, dobara mat karna
model = SelectFromModel(estimator, max_features=3, prefit=True)

# 4. Filter apply kiya (Sirf X pass karte hain kyunki model pehle hi train ho chuka hai)
X_new = model.transform(X)

print(f'\nShape of feature matrix after feature selection: {X_new.shape}')


Coefficients of features:
[ 7.02844704  5.83051413 -1.78345466 11.44136077  3.45620539 -1.73646096
  1.5900031  -1.91542332 -0.572441    1.99475797]

Shape of feature matrix after feature selection: (50, 3)


### 2.3 🧩 Feature Selection: `SequentialFeatureSelector` (SFS)

## 🤔 1. Ye kya hai aur "Greedy" hone ka kya matlab hai?

**Sequential Feature Selection (SFS)** ek Wrapper method hai. Yeh features ko ek-ek karke apne team (dataset) me jodta hai ya nikalta hai. 

* **"Greedy Manner" (Lalchi Tarika):** Iska matlab hai ki yeh machine har step par sirf apna "aaj" dekhti hai, "kal" nahi. 
  * *Example:* Agar isko 1 feature chunna hai, toh yeh sabse best chun legi. Phir jab 2nd chunna hoga, toh yeh dekhegi ki pehle wale ke sath kaun sabse best jodi (pair) bana raha hai. Isko is baat se matlab nahi hai ki aage chalkar koi aur combination behtar ho sakta tha. Jo current step me best hai, bas wahi lock kar do!

---

## 🔄 2. The Two Modes: Forward vs Backward

Tumhare code me do alag-alag outputs hain kyunki SFS do alag dishaon (directions) me kaam karta hai:

### A. Forward Selection (Default Mode)
* **Start:** Zero columns.
* **Process:** Pehle sabse best column chunega. Phir dusra best chunega jo pehle wale ke sath achha chale. Phir teesra.
* **Stop:** Jab tak target (jaise `k=3`) poora na ho jaye.

### B. Backward Selection (`direction='backward'`)
* **Start:** Saare ke saare columns (8 ke 8).
* **Process:** Pehle us column ko dhundhega jo sabse bekar hai aur usko nikal fekega (Drop). Phir bache hue 7 me se sabse bekar ko nikalega.
* **Stop:** Jab tak sirf target (`k=3`) columns na bach jayein.

---


In [53]:
import numpy as np
import time
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

# ---------------------------------------------------------
# 1. DATA PREPARATION
# ---------------------------------------------------------
# Ek nakli dataset banate hain jisme 500 rows aur 8 columns (features) hain
X, y = make_regression(n_samples=500, n_features=8, noise=0.1, random_state=42)

print(f"Original Dataset Shape: {X.shape}\n")
print("=" * 50)

# ---------------------------------------------------------
# 2. FORWARD SELECTION (Greedy - Starts from 0 features)
# ---------------------------------------------------------
print("🚀 Running Forward Selection...")
start_time_fwd = time.time()

estimator_fwd = LinearRegression()
# Default direction 'forward' hi hoti hai
sfs_forward = SequentialFeatureSelector(estimator_fwd, n_features_to_select=3, direction='forward')
X_new_fwd = sfs_forward.fit_transform(X, y)

end_time_fwd = time.time()

print("Forward Selection Support (True means selected):")
print(sfs_forward.get_support())
print(f"Time Taken (Forward): {round((end_time_fwd - start_time_fwd) * 1000, 2)} ms\n")
print("=" * 50)

# ---------------------------------------------------------
# 3. BACKWARD SELECTION (Greedy - Starts from all 8 features)
# ---------------------------------------------------------
print("🐢 Running Backward Selection...")
start_time_bwd = time.time()

estimator_bwd = LinearRegression()
# Is baar direction 'backward' set ki hai
sfs_backward = SequentialFeatureSelector(estimator_bwd, n_features_to_select=3, direction='backward')
X_new_bwd = sfs_backward.fit_transform(X, y)

end_time_bwd = time.time()

print("Backward Selection Support (True means selected):")
print(sfs_backward.get_support())
print(f"Time Taken (Backward): {round((end_time_bwd - start_time_bwd) * 1000, 2)} ms\n")
print("=" * 50)

Original Dataset Shape: (500, 8)

🚀 Running Forward Selection...
Forward Selection Support (True means selected):
[False False  True False  True False  True False]
Time Taken (Forward): 79.65 ms

🐢 Running Backward Selection...
Backward Selection Support (True means selected):
[False False  True False  True False  True False]
Time Taken (Backward): 92.14 ms



## 3. PCA

In [60]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 2)
pca.fit(X)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",2
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 